In [ ]:
import os
import psutil
from PIL import Image
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path


def process_image(args):
    """Process one image: duplicate pixels using PIL's efficient resizing."""
    input_folder, output_folder, filename, scale_factor = args
    
    try:
        img_path = os.path.join(input_folder, filename)
        with Image.open(img_path) as img:
            # Use PIL's NEAREST neighbor - much faster than NumPy for pixel duplication
            new_size = (img.width * scale_factor, img.height * scale_factor)
            upscaled_img = img.resize(new_size, Image.Resampling.NEAREST)
            
            # Compute new DPI metadata
            orig_dpi = img.info.get('dpi', (72, 72))
            new_dpi = (orig_dpi[0] * scale_factor, orig_dpi[1] * scale_factor)
            
            out_path = os.path.join(output_folder, filename)
            upscaled_img.save(out_path, dpi=new_dpi)
            
            return filename, new_size, new_dpi, None
    except Exception as e:
        return filename, None, None, str(e)


def get_image_files(folder):
    """Get all image files from folder."""
    extensions = {'.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.webp'}
    return [
        f for f in os.listdir(folder)
        if Path(f).suffix.lower() in extensions
    ]


def estimate_memory_per_image(sample_path, scale_factor):
    """Estimate memory needed per image based on a sample."""
    try:
        with Image.open(sample_path) as img:
            # Rough estimate: original + upscaled + overhead
            orig_size = img.width * img.height * len(img.getbands())
            upscaled_size = orig_size * (scale_factor ** 2)
            return (orig_size + upscaled_size) * 2  # 2x safety margin
    except:
        return 100 * 1024 * 1024  # Default 100MB if can't estimate


def increase_dpi_by_duplication(
    input_folder,
    output_folder,
    scale_factor=2,
    max_workers=None,
    reserved_cores=2,
    max_memory_percent=85,  # Use up to 85% of available memory
    progress_interval=100
):
    """
    Efficiently increases DPI by duplicating pixels using multiprocessing.
    
    Args:
        input_folder: Source image directory
        output_folder: Output directory
        scale_factor: Multiplier for both dimensions
        max_workers: Max processes (None = auto-detect)
        reserved_cores: CPU cores to keep free
        max_memory_percent: Max memory usage percentage
        progress_interval: Show progress every N images
    """
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all image files
    all_files = get_image_files(input_folder)
    total_images = len(all_files)
    
    if total_images == 0:
        print(f"No images found in '{input_folder}'")
        return
    
    print(f"Found {total_images} images in '{input_folder}'")
    
    # Determine optimal worker count
    cpu_count = os.cpu_count() or 1
    usable_cores = max(1, cpu_count - reserved_cores)
    
    if max_workers is None:
        max_workers = usable_cores
    else:
        max_workers = min(max_workers, usable_cores)
    
    # Estimate memory constraints
    mem = psutil.virtual_memory()
    available_memory = mem.available * (max_memory_percent / 100)
    
    sample_path = os.path.join(input_folder, all_files[0])
    mem_per_image = estimate_memory_per_image(sample_path, scale_factor)
    
    # Limit workers based on memory
    memory_limited_workers = max(1, int(available_memory / mem_per_image))
    max_workers = min(max_workers, memory_limited_workers)
    
    print(f"Using {max_workers} workers (CPU cores: {cpu_count}, memory-limited: {memory_limited_workers})")
    print(f"Available memory: {available_memory / (1024**3):.1f}GB, Est. per image: {mem_per_image / (1024**2):.1f}MB")
    
    # Process images
    completed = 0
    errors = []
    
    # Prepare arguments for each image
    tasks = [(input_folder, output_folder, f, scale_factor) for f in all_files]
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_image, task): task[2] for task in tasks}
        
        for future in as_completed(futures):
            filename, size, dpi, error = future.result()
            completed += 1
            
            if error:
                errors.append((filename, error))
                print(f"❌ Error processing {filename}: {error}")
            
            # Progress update
            if completed % progress_interval == 0 or completed == total_images:
                mem_used = psutil.virtual_memory().percent
                print(f"✅ Processed {completed}/{total_images} images (Memory: {mem_used:.1f}%)")
    
    # Summary
    print(f"\n🎉 Completed: {completed - len(errors)}/{total_images} successful")
    if errors:
        print(f"⚠️  Errors: {len(errors)} failed")
        for fname, err in errors[:5]:  # Show first 5 errors
            print(f"   - {fname}: {err}")


if __name__ == "__main__":
    increase_dpi_by_duplication(
        input_folder="/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60/",
        output_folder="/home/ajarrah/PhD_Thesis/chapter_4/images/genes/lipids_60_to_20/",
        scale_factor=3,
        max_workers=None,  # Auto-detect
        reserved_cores=2,
        max_memory_percent=85,
        progress_interval=100
    )

No images found in '/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60/'


In [1]:
#===============================================================================
#                  Process images in subdirectories
#===============================================================================

import os
import psutil
from PIL import Image
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path


def process_image(args):
    """Process one image: duplicate pixels using PIL's efficient resizing."""
    input_folder, output_folder, rel_path, scale_factor = args
    
    try:
        img_path = os.path.join(input_folder, rel_path)
        with Image.open(img_path) as img:
            # Use PIL's NEAREST neighbor - much faster than NumPy for pixel duplication
            new_size = (img.width * scale_factor, img.height * scale_factor)
            upscaled_img = img.resize(new_size, Image.Resampling.NEAREST)
            
            # Compute new DPI metadata
            orig_dpi = img.info.get('dpi', (72, 72))
            new_dpi = (orig_dpi[0] * scale_factor, orig_dpi[1] * scale_factor)
            
            # Create output path maintaining folder structure
            out_path = os.path.join(output_folder, rel_path)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            upscaled_img.save(out_path, dpi=new_dpi)
            
            return rel_path, new_size, new_dpi, None
    except Exception as e:
        return rel_path, None, None, str(e)


def get_image_files(folder):
    """Get all image files from folder and subfolders recursively."""
    extensions = {'.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.webp'}
    image_files = []
    
    for root, dirs, files in os.walk(folder):
        for filename in files:
            if Path(filename).suffix.lower() in extensions:
                # Store relative path from input folder
                rel_path = os.path.relpath(os.path.join(root, filename), folder)
                image_files.append(rel_path)
    
    return image_files


def estimate_memory_per_image(sample_path, scale_factor):
    """Estimate memory needed per image based on a sample."""
    try:
        with Image.open(sample_path) as img:
            # Rough estimate: original + upscaled + overhead
            orig_size = img.width * img.height * len(img.getbands())
            upscaled_size = orig_size * (scale_factor ** 2)
            return (orig_size + upscaled_size) * 2  # 2x safety margin
    except:
        return 100 * 1024 * 1024  # Default 100MB if can't estimate


def increase_dpi_by_duplication(
    input_folder,
    output_folder,
    scale_factor=2,
    max_workers=None,
    reserved_cores=2,
    max_memory_percent=85,  # Use up to 85% of available memory
    progress_interval=100
):
    """
    Efficiently increases DPI by duplicating pixels using multiprocessing.
    
    Args:
        input_folder: Source image directory
        output_folder: Output directory
        scale_factor: Multiplier for both dimensions
        max_workers: Max processes (None = auto-detect)
        reserved_cores: CPU cores to keep free
        max_memory_percent: Max memory usage percentage
        progress_interval: Show progress every N images
    """
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all image files
    all_files = get_image_files(input_folder)
    total_images = len(all_files)
    
    if total_images == 0:
        print(f"No images found in '{input_folder}'")
        return
    
    print(f"Found {total_images} images in '{input_folder}' (including subdirectories)")
    
    # Determine optimal worker count
    cpu_count = os.cpu_count() or 1
    usable_cores = max(1, cpu_count - reserved_cores)
    
    if max_workers is None:
        max_workers = usable_cores
    else:
        max_workers = min(max_workers, usable_cores)
    
    # Estimate memory constraints
    mem = psutil.virtual_memory()
    available_memory = mem.available * (max_memory_percent / 100)
    
    sample_path = os.path.join(input_folder, all_files[0])
    mem_per_image = estimate_memory_per_image(sample_path, scale_factor)
    
    # Limit workers based on memory
    memory_limited_workers = max(1, int(available_memory / mem_per_image))
    max_workers = min(max_workers, memory_limited_workers)
    
    print(f"Using {max_workers} workers (CPU cores: {cpu_count}, memory-limited: {memory_limited_workers})")
    print(f"Available memory: {available_memory / (1024**3):.1f}GB, Est. per image: {mem_per_image / (1024**2):.1f}MB")
    
    # Process images
    completed = 0
    errors = []
    
    # Prepare arguments for each image
    tasks = [(input_folder, output_folder, f, scale_factor) for f in all_files]
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_image, task): task[2] for task in tasks}
        
        for future in as_completed(futures):
            filename, size, dpi, error = future.result()
            completed += 1
            
            if error:
                errors.append((rel_path, error))
                print(f"❌ Error processing {rel_path}: {error}")
            
            # Progress update
            if completed % progress_interval == 0 or completed == total_images:
                mem_used = psutil.virtual_memory().percent
                print(f"✅ Processed {completed}/{total_images} images (Memory: {mem_used:.1f}%)")
    
    # Summary
    print(f"\n🎉 Completed: {completed - len(errors)}/{total_images} successful")
    if errors:
        print(f"⚠️  Errors: {len(errors)} failed")
        for fname, err in errors[:5]:  # Show first 5 errors
            print(f"   - {fname}: {err}")


if __name__ == "__main__":
    increase_dpi_by_duplication(
        input_folder="/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60_common/",
        output_folder="/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60_to_20_common/",
        scale_factor=3,
        max_workers=None,  # Auto-detect
        reserved_cores=2,
        max_memory_percent=85,
        progress_interval=100
    )

Found 8448 images in '/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60_common/' (including subdirectories)
Using 78 workers (CPU cores: 80, memory-limited: 479066)
Available memory: 152.7GB, Est. per image: 0.3MB
✅ Processed 100/8448 images (Memory: 4.5%)
✅ Processed 200/8448 images (Memory: 4.5%)
✅ Processed 300/8448 images (Memory: 4.5%)
✅ Processed 400/8448 images (Memory: 4.5%)
✅ Processed 500/8448 images (Memory: 4.5%)
✅ Processed 600/8448 images (Memory: 4.5%)
✅ Processed 700/8448 images (Memory: 4.5%)
✅ Processed 800/8448 images (Memory: 4.5%)
✅ Processed 900/8448 images (Memory: 4.5%)
✅ Processed 1000/8448 images (Memory: 4.5%)
✅ Processed 1100/8448 images (Memory: 4.5%)
✅ Processed 1200/8448 images (Memory: 4.5%)
✅ Processed 1300/8448 images (Memory: 4.5%)
✅ Processed 1400/8448 images (Memory: 4.5%)
✅ Processed 1500/8448 images (Memory: 4.5%)
✅ Processed 1600/8448 images (Memory: 4.5%)
✅ Processed 1700/8448 images (Memory: 4.5%)
✅ Processed 1800/8448 images (Memory: 4.5%)
✅